
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/05_oop/05_oop.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 05 — Object-Oriented Programming (OOP)

**Learning Objectives:** Classes, instances, attributes, methods, inheritance, encapsulation, magic methods

**Estimated time:** 60–90 minutes

---

## 5.1 What Is Object-Oriented Programming?

**The core idea:**
OOP is a way of organising code by grouping related *data* and *behaviour* together into a single unit called a **class**. Instead of having separate variables for a student's name, age, and grades floating around your code, OOP lets you bundle them into one `Student` object that also knows how to *do things* with that data.

**The real-world analogy:**
Think of a **class** as a blueprint — like an architect's drawing for a house. The blueprint defines what every house will have (rooms, doors, windows) and what it can do (open door, turn on lights). An **instance** (or object) is an actual house built from that blueprint. You can build many houses from one blueprint, and each house has its own address, furniture, and state — but they all follow the same structure.

**Why OOP matters in data science and ML:**
- Scikit-learn models (`RandomForestClassifier`, `LogisticRegression`) are all classes
- Pandas DataFrames are objects with methods like `.groupby()`, `.merge()`
- PyTorch and Keras neural networks are built as classes
- Understanding OOP lets you *read* and *extend* these tools, not just use them

**Four pillars of OOP:**
1. **Encapsulation** — bundle data and behaviour together, hide internal details
2. **Inheritance** — a child class reuses and extends a parent class
3. **Polymorphism** — different classes can share the same interface
4. **Abstraction** — expose only what is necessary, hide complexity

## 5.2 Classes and Instances

**What is a class?**
A class is defined using the `class` keyword. It is a template that describes:
- What **data** (attributes) each object will store
- What **actions** (methods) each object can perform

**What is `__init__`?**
`__init__` is a special method called the **constructor** or **initialiser**. Python calls it automatically every time you create a new instance. It is where you set up the object's initial state by assigning values to `self.attribute_name`.

**What is `self`?**
`self` is a reference to the specific instance being created or used. When you call `dog.bark()`, Python passes `dog` as `self` automatically. Every method must have `self` as its first parameter — this is how the method knows *which* object's data to work with.

**Class attributes vs instance attributes:**
- **Class attributes** are defined directly in the class body (outside any method). They are shared by ALL instances.
- **Instance attributes** are defined inside `__init__` using `self.name = value`. Each instance has its own copy.

In [ ]:
class Dog:
    # Class attribute — shared by ALL Dog instances
    species = "Canis lupus familiaris"
    
    def __init__(self, name, breed, age):
        # Instance attributes — unique to EACH Dog object
        self.name  = name
        self.breed = breed
        self.age   = age
    
    def bark(self):
        # 'self' gives access to this specific dog's attributes
        return f"{self.name} says: Woof!"
    
    def describe(self):
        return f"{self.name} is a {self.age}-year-old {self.breed}"
    
    def have_birthday(self):
        self.age += 1    # modifies THIS dog's age
        return f"Happy birthday {self.name}! Now {self.age} years old."
    
    def __repr__(self):
        # Controls how the object looks when printed or in a REPL
        return f"Dog(name='{self.name}', breed='{self.breed}', age={self.age})"

# Creating instances — each is independent
rex  = Dog("Rex",  "German Shepherd", 3)
luna = Dog("Luna", "Labrador",        5)

print(rex.bark())
print(luna.describe())
print(rex.have_birthday())

# Class attribute is accessible on any instance
print(rex.species)
print(luna.species)
print(Dog.species)   # also accessible on the class itself

print(rex)    # uses __repr__

**Key insight — what just happened:**
When you wrote `rex = Dog("Rex", "German Shepherd", 3)`, Python:
1. Created a new empty object in memory
2. Called `Dog.__init__(rex, "Rex", "German Shepherd", 3)` automatically
3. `__init__` set `rex.name = "Rex"`, `rex.breed = "German Shepherd"`, `rex.age = 3`
4. Returned the fully initialised object and assigned it to `rex`

`rex` and `luna` are completely independent — changing `rex.age` has no effect on `luna.age`.

## 5.3 Methods in Depth

**What is a method?**
A method is simply a function that belongs to a class. The only difference from a regular function is that it receives `self` as its first argument, giving it access to the object's data.

**Types of methods:**
- **Instance methods** — the most common. Take `self`, operate on instance data
- **@property** — lets you access a method *as if it were an attribute* (no parentheses). Use this for computed values or read-only attributes
- **@classmethod** — receives the class itself instead of an instance. Used for alternative constructors
- **@staticmethod** — no `self` or `cls`. Just a regular function grouped inside a class for organisation

**The `_` naming convention:**
A single underscore prefix (`_balance`) is a Python convention meaning "this is intended to be private — please don't access it directly from outside the class." Python does not enforce this, but it is a signal to other developers.

**Why use `@property` instead of direct access?**
It lets you add validation, computation, or logging behind what looks like a simple attribute access — without changing the interface for users of your class.

In [ ]:
class BankAccount:
    
    def __init__(self, owner, balance=0):
        self.owner    = owner
        self._balance = balance   # _ means: treat as private, use .balance property instead
    
    # --- Instance methods ---
    def deposit(self, amount):
        """Add money to the account."""
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self._balance += amount
        print(f"  Deposited £{amount:,.2f}. New balance: £{self._balance:,.2f}")
    
    def withdraw(self, amount):
        """Remove money from the account."""
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self._balance:
            raise ValueError(f"Insufficient funds. Balance: £{self._balance:,.2f}")
        self._balance -= amount
        print(f"  Withdrew £{amount:,.2f}. New balance: £{self._balance:,.2f}")
    
    # --- Property: access _balance safely without parentheses ---
    @property
    def balance(self):
        """Read-only access to the balance."""
        return self._balance
    
    # --- Class method: alternative constructor ---
    @classmethod
    def from_dict(cls, data):
        """Create an account from a dictionary."""
        return cls(data["owner"], data.get("balance", 0))
    
    def __str__(self):
        return f"BankAccount(owner='{self.owner}', balance=£{self._balance:,.2f})"

# Using the class
acc = BankAccount("Alice", 1000)
print(acc)

acc.deposit(500)
acc.withdraw(200)

# @property: accessed like an attribute, no ()
print(f"Current balance: £{acc.balance:,.2f}")

# acc._balance = 9999  # technically works but breaks the convention — don't do this
# acc.balance = 9999   # this raises AttributeError — truly read-only via @property

# Alternative constructor
acc2 = BankAccount.from_dict({"owner": "Bob", "balance": 2500})
print(acc2)

## 5.4 Inheritance

**What is inheritance?**
Inheritance lets one class (the **child** or **subclass**) automatically receive all the attributes and methods of another class (the **parent** or **superclass**). The child can then:
- Use the parent's methods as-is
- **Override** a method to give it different behaviour
- **Extend** a method by calling `super()` to run the parent's version first, then adding more

**Why use inheritance?**
The DRY principle — Don't Repeat Yourself. If five animal types all need a `name` attribute and a `__str__` method, define those once in `Animal` and inherit them in all five subclasses.

**What is `super()`?**
`super()` refers to the parent class. Calling `super().__init__(...)` runs the parent's `__init__` so you don't have to repeat that setup code in every child class.

**What is polymorphism?**
When different child classes all implement the same method name (like `speak()`), you can loop over a list of mixed animal types and call `.speak()` on each one — Python automatically calls the right version for each type. This is polymorphism: same interface, different behaviour.

In [ ]:
# Parent class — defines what ALL animals share
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age  = age
    
    def speak(self):
        # Subclasses MUST override this — it makes no sense for generic Animal
        raise NotImplementedError(f"{self.__class__.__name__} must implement speak()")
    
    def __str__(self):
        return f"{self.__class__.__name__}(name='{self.name}', age={self.age})"


# Child classes — inherit from Animal, override speak()
class Dog(Animal):
    def __init__(self, name, age, breed):
        super().__init__(name, age)   # run Animal's __init__ first
        self.breed = breed            # then add Dog-specific attribute
    
    def speak(self):
        return f"{self.name} says: Woof!"
    
    def fetch(self):
        return f"{self.name} fetches the ball!"


class Cat(Animal):
    def speak(self):
        return f"{self.name} says: Meow!"


class Duck(Animal):
    def speak(self):
        return f"{self.name} says: Quack!"


# Polymorphism in action — same code works for ALL animal types
animals = [
    Dog("Rex",     3, "German Shepherd"),
    Cat("Whiskers", 5),
    Duck("Donald",  2),
    Dog("Luna",    4, "Labrador"),
]

print("All animals speaking (polymorphism):")
for animal in animals:
    print(f"  {animal.speak()}")   # Python calls the RIGHT speak() for each type

print()
print("Only dogs can fetch:")
for animal in animals:
    if isinstance(animal, Dog):    # isinstance() checks the type
        print(f"  {animal.fetch()}")

print()
print("Inheritance check:")
print(isinstance(Rex := Dog("Rex", 3, "GSD"), Dog))      # True
print(isinstance(Rex, Animal))                            # True — Dog IS an Animal
print(isinstance(Rex, Cat))                               # False

## 5.5 Magic (Dunder) Methods

**What are dunder methods?**
Dunder methods (short for "double underscore") are special methods like `__init__`, `__str__`, `__add__` that Python calls automatically in specific situations. They allow your custom classes to behave like Python's built-in types.

**Why do they matter?**
When you write `a + b`, Python actually calls `a.__add__(b)`. When you write `print(obj)`, Python calls `obj.__str__()`. When you write `len(obj)`, Python calls `obj.__len__()`. By implementing these methods in your class, you make your objects feel native to Python.

**Common dunder methods:**

| Method | When Python calls it | Example trigger |
|--------|---------------------|-----------------|
| `__init__` | Object creation | `Dog("Rex", 3)` |
| `__str__` | Human-readable string | `print(obj)`, `str(obj)` |
| `__repr__` | Developer string | REPL display, `repr(obj)` |
| `__len__` | Length | `len(obj)` |
| `__add__` | Addition | `obj1 + obj2` |
| `__eq__` | Equality | `obj1 == obj2` |
| `__lt__` | Less than | `obj1 < obj2` |
| `__getitem__` | Index access | `obj[0]` |
| `__iter__` | Iteration | `for x in obj` |

In [ ]:
class Vector:
    """
    A 2D mathematical vector.
    Demonstrates how dunder methods make custom objects feel like built-in types.
    """
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        # Used in REPL and when printing lists of vectors
        return f"Vector({self.x}, {self.y})"
    
    def __str__(self):
        # Used by print() — more human-friendly
        return f"({self.x}, {self.y})"
    
    def __add__(self, other):
        # Called when you write: v1 + v2
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        # Called when you write: v1 - v2
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):
        # Called when you write: v * 3
        return Vector(self.x * scalar, self.y * scalar)
    
    def __eq__(self, other):
        # Called when you write: v1 == v2
        return self.x == other.x and self.y == other.y
    
    def __abs__(self):
        # Called when you write: abs(v) — returns the vector's magnitude
        return (self.x**2 + self.y**2) ** 0.5
    
    def __len__(self):
        # Called when you write: len(v) — returns integer magnitude
        return int(abs(self))

# Using the Vector class — notice how natural the syntax is
v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"v1 + v2 = {v1 + v2}")     # calls __add__
print(f"v2 - v1 = {v2 - v1}")     # calls __sub__
print(f"v1 * 3  = {v1 * 3}")      # calls __mul__
print(f"v1 == v1: {v1 == v1}")    # calls __eq__
print(f"|v2| = {abs(v2):.2f}")    # calls __abs__   (magnitude = 5.0)
print(f"len(v2) = {len(v2)}")     # calls __len__

# Lists of vectors work naturally now
vectors = [Vector(3,4), Vector(1,1), Vector(0,5)]
print("Sorted by magnitude:", sorted(vectors, key=abs))

---

## Key Takeaways

- A **class** is a blueprint. An **instance** is an object built from that blueprint.
- `__init__` sets up each object's initial data. `self` refers to the specific instance.
- **Instance attributes** (`self.x`) belong to one object. **Class attributes** belong to all.
- `@property` lets you access a method like an attribute — great for read-only values.
- **Inheritance** lets child classes reuse parent code. `super()` calls the parent's method.
- **Polymorphism** means different classes can share the same method name with different behaviour.
- **Dunder methods** make your objects behave like Python built-ins (`+`, `len()`, `print()`).

## Exercises

[05_exercises.ipynb](exercises/05_exercises.ipynb) | [05_solutions.ipynb](exercises/05_solutions.ipynb)

## Next: [06 — Modules & Packages](../06_modules_packages/06_modules_and_packages.ipynb)
